<a href="https://colab.research.google.com/github/Angel-ag-1/ML-pipeline/blob/main/capstone_By_Angel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/Angel-ag-1/ML-pipeline.git"
REPO_DIR = "ML-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URL,
                REPO_DIR
            ],
            check=True
        )

    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/ML-pipeline/ML-pipeline


In [ ]:
!pip -q install pandas pyarrow huggingface_hub scikit-learn matplotlib

In [ ]:
from huggingface_hub import (
    login,
    whoami,
    hf_hub_download
)

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

login(token=HF_TOKEN)

print(
    "Hugging Face account:",
    whoami()["name"]
)

Hugging Face account: angelhi


In [ ]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    )
)

march = pd.read_parquet(march_path)

print("March data shape:", march.shape)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March data shape: (9841378, 30)


In [ ]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    )
)

march = pd.read_parquet(march_path)

print("March data shape:", march.shape)

March data shape: (9841378, 30)


In [ ]:
march_agg = (
    march
    .groupby(
        [
            "client_hash_id",
            "content_hash_id"
        ]
    )
    .agg(
        gsc_impressions=(
            "gsc_impressions",
            "sum"
        ),
        gsc_clicks=(
            "gsc_clicks",
            "sum"
        ),
        gsc_avg_position=(
            "gsc_avg_position",
            "mean"
        ),
        ga4_sessions=(
            "ga4_sessions",
            "sum"
        ),
        ga4_pageviews=(
            "ga4_pageviews",
            "sum"
        )
    )
    .reset_index()
)

march_agg["ctr"] = (
    march_agg["gsc_clicks"]
    /
    march_agg["gsc_impressions"].replace(
        0,
        pd.NA
    )
)

march_agg["ctr"] = (
    march_agg["ctr"].fillna(0)
)

print("Webpages:", len(march_agg))
print(
    "Clients:",
    march_agg["client_hash_id"].nunique()
)

march_agg.head()

Webpages: 331437
Clients: 55


/tmp/ipykernel_1275/3157809512.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  march_agg["ctr"].fillna(0)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_pageviews,ctr
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,0.0,0.0
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,0.0,0.0
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,0.0,0.0
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,0.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,0.0,0.0


## 1. Question

*The research question and the decision it supports.*

This project investigates whether March 2026 search and website performance signals can help prioritise webpages for manual content review.

The decision supported is which webpages should be reviewed first. The results are intended as decision and not as automatic content decisions.

In [ ]:
print("=" * 70)
print("RESEARCH QUESTION")
print("=" * 70)

print(
    "Can March 2026 search and website performance signals help "
    "prioritise webpages for manual content review?"
)

print()
print("Decision supported:")
print(
    "Which webpages should be prioritised for human review?"
)

print()
print(
    "This project provides decision-support only."
)

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the FlyRank ML Internship dataset and March 2026 daily content performance data.

The data is aggregated by client and webpage. Client names, webpage URLs and private queries are excluded from the analysis.

In [ ]:
print("=" * 70)
print("DATA SUMMARY")
print("=" * 70)

print("Dataset: FlyRank ML Internship dataset")
print(
    "Table: fact_content_daily_performance"
)
print("Date window: March 2026")

print()
print("Rows:", len(march))
print("Aggregated webpages:", len(march_agg))
print(
    "Unique clients:",
    march_agg["client_hash_id"].nunique()
)

print()
print("Excluded:")
print("- Client names")
print("- Webpage URLs")
print("- Private search queries")

print()
print(
    "Public-safe identifiers were used where necessary."
)

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

A Decision Tree was used because it provides an interpretable comparison.

The target is `needs_attention`, defined as webpages with zero March impressions. The model uses search clicks, average search position, sessions, and pageviews.

Validation is grouped by client to reduce overlap between training and testing clients. Leakage checks exclude identifiers and the impressions feature that directly defines the target.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
